# Attempt 003 — prompted general-model judge

Frozen public Gemma 4, fixed neutral prompt, no model fine-tuning and no classifier
ensemble. Three single-token labels map to A, B and tie. The quantized model and
compiled inference runtime are versioned Kaggle inputs; submission internet is off.
See `docs/ATTEMPT_003.md` for the predeclared evaluation and runtime protocol.


In [1]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
repo=Path.cwd()
if repo.name=='notebooks':repo=repo.parent
directory=repo/'results/attempt_003'
status=json.loads((directory/'status.json').read_text())
print('Pipeline:',status['state']['stage'])
print('Model weights trained:',status['weights_trained'])
print('Blended with classifier:',status['blended_with_classifier'])
display(status.get('official_submission'))


Pipeline: failed
Model weights trained: False
Blended with classifier: False


None

The fixed 512-row development sample uses seed 20261113 and was selected
without looking at judge errors. Labels never enter judge inputs. Both orientations
are scored where feasible; reversed labels are aligned before probability averaging.
The 2,048-token packing preserves question and both answers with head/tail truncation.
Development has been used before; results are exploratory rather than fresh confirmation.


In [2]:
path=directory/'development.json'
if path.exists():
    development=json.loads(path.read_text())
    rows=[]
    for r in development['rows']:
        rows.append({'model':r['model_key'],'view':r['view'],'calibration':r['calibration'],**r['metrics'],
                     'gain_vs_lora':r['paired_gain_vs_lora'],'gain_95pct_interval':r['paired_gain_interval'],
                     'runtime_hours_with_margin':r['projected_with_margin_seconds']/3600,
                     'runtime_feasible':r['runtime_feasible']})
    table=pd.DataFrame(rows)
    display(table.drop(columns=['confusion_matrix']).round(5))
    baseline=development['rows'][0]['baseline_same_rows']
    print('Frozen LoRA on exactly the same IDs:',baseline)
    display(development['constant_baselines'])
    names=[f"{r['model']} / {r['view']} / {r['calibration']}" for r in rows]
    fig,ax=plt.subplots(figsize=(10,max(4,len(rows)*.35)))
    ax.barh(names,[r['log_loss'] for r in rows],color='#26796c')
    ax.axvline(baseline['log_loss'],color='#a45939',linestyle='--',label='Frozen Qwen3-4B on same rows')
    ax.invert_yaxis();ax.set_xlabel('Log loss (lower is better)');ax.legend();fig.tight_layout();plt.show()
else:
    print('Development inference is pending; this notebook will be refreshed automatically.')


Development inference is pending; this notebook will be refreshed automatically.


Temperature and optional tie-intercept calibration use five folds grouped
by prompt. Their fitted parameters are separate from model-weight training. Selection
among configurations still uses development. Bootstrap intervals are descriptive and
do not undo prior model selection. The final chosen calibration is fitted on the full
sample only after this comparison.

Runtime estimates assume 25,000 hidden rows, add a 20% margin and 600 seconds, and
must not be mistaken for observed hidden-test time. The final notebook reads actual
test rows. A job failure is reported rather than silently filling rows with priors.


In [3]:
selection=status.get('selection')
if selection:
    display({k:v for k,v in selection.items() if k not in ['config','development']})
    print('Estimated selected development loss:',selection['development']['metrics']['log_loss'])
else:print('Selection not frozen yet.')
official=status.get('official_submission')
if official:display(official)


Selection not frozen yet.
